<a href="https://colab.research.google.com/github/meryambutt123-a11y/urdu-ocr-codesaviours-si26-maryam/blob/main/SI26-Week2-Maryam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 2: Image Preprocessing and Baseline Tesseract Testing on Urdu Nastaliq Script**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install opencv-python-headless pillow
import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
print('Libraries loaded successfully!')

Libraries loaded successfully!


In [12]:
# Essential imports for a fresh session
import cv2
import os

# Cell 2 - Core Preprocessing Pipeline Function
def preprocess_image(image_path, save_path):
    # Load the raw image using OpenCV
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return None

    # Step 1: Convert to grayscale (removes RGB color noise)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Resize to uniform dimensions (crucial for neural network inputs)
    resized = cv2.resize(gray, (512, 128))

    # Step 3: Non-Local Means Denoising (Reduced 'h' to 5 to protect thin book text)
    denoised = cv2.fastNlMeansDenoising(resized, h=5)

    # Step 4: Otsu's Binarization (Automatically calculates the perfect threshold per image)
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Save the processed image into the output folder
    cv2.imwrite(save_path, binary)
    return binary

# Create the dedicated output directory structure if it doesn't exist
os.makedirs('data/processed', exist_ok=True)
print('Preprocessing function ready with Otsu!')

Preprocessing function ready with Otsu!


In [13]:
import os

print("Searching your Google Drive for the data folder...")
for root, dirs, files in os.walk('/content/drive/MyDrive/'):
    if 'newspaper' in dirs and 'books' in dirs:
        # This looks for the exact folder containing your subdirectories
        target_dir = os.path.dirname(root)
        print(f"\n🎯 Found it! Your true project path is: {target_dir}")
        print(f"Copy and run this exact command: %cd {target_dir}")
        break
else:
    print("\nCould not find the 'newspaper' and 'books' folders. Are you sure they are on this Google Drive account?")

Searching your Google Drive for the data folder...

🎯 Found it! Your true project path is: /content/drive/MyDrive/Urdu_OCR_Project/data
Copy and run this exact command: %cd /content/drive/MyDrive/Urdu_OCR_Project/data


In [7]:
%cd /content/drive/MyDrive/Urdu_OCR_Project/data

/content/drive/MyDrive/Urdu_OCR_Project/data


In [14]:
# Cell 3 - Run Preprocessing on All Images
%cd /content/drive/MyDrive/Urdu_OCR_Project

import glob
import os

# THE FIX: Force Google Drive to physically create the folder first
os.makedirs('data/processed', exist_ok=True)

# Dynamically locate all raw images (JPG and PNG) across all subfolders
all_images = glob.glob('data/raw/**/*.jpg', recursive=True)
all_images += glob.glob('data/raw/**/*.png', recursive=True)

print(f'Found {len(all_images)} images to process...')

# Process each image through the pipeline
processed_count = 0
for img_path in all_images:
    filename = os.path.basename(img_path)
    save_path = f'data/processed/{filename}'

    result = preprocess_image(img_path, save_path)
    if result is not None:
        processed_count += 1

print(f'Done! Successfully processed {processed_count} images.')
print('Check your data/processed/ folder to see the results!')

/content/drive/MyDrive/Urdu_OCR_Project
Found 139 images to process...
Done! Successfully processed 139 images.
Check your data/processed/ folder to see the results!


In [15]:
# Cell 4 - Install and Test Tesseract Engine

# 1. Install the core Tesseract engine and the Urdu language pack to the Linux environment
!apt-get install -y tesseract-ocr tesseract-ocr-urd

# 2. Install the PyTesseract Python wrapper
!pip install pytesseract

import pytesseract
from PIL import Image
import glob
import os

# 3. Pull 5 sample images from your freshly processed folder
# Ensure we are in the correct directory first
%cd /content/drive/MyDrive/Urdu_OCR_Project

test_images = glob.glob('data/processed/*.png')[:5]

print('\n=== Tesseract Results on Processed Urdu Images ===\n')

if not test_images:
    print("No images found in data/processed/. Did you run yesterday's batch loop successfully?")
else:
    for img_path in test_images:
        img = Image.open(img_path)

        # 'urd' explicitly tells Tesseract to look for Urdu characters and layout rules
        result = pytesseract.image_to_string(img, lang='urd')

        print(f'Image: {os.path.basename(img_path)}')
        print(f'Tesseract Raw Output:\n{result}')
        print('--------------------------------------------\n')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-urd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 1,000 kB of archives.
After this operation, 1,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-urd all 1:4.00~git30-7274cfa-1.1 [1,000 kB]
Fetched 1,000 kB in 1s (985 kB/s)
Selecting previously unselected package tesseract-ocr-urd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-urd_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
/content/drive/MyDrive/Urdu_OCR_Project

=== Tesseract Results on Processed Urdu Images ===

Image: newspaper_17.png
Tesser